# build_vocab: the PolyPlex v2 text vocabulary

This notebook decides the final text vocabulary (SPEC S2.3, S12.1a, S12.4, S12.6). Running it top to bottom produces the
same vocabulary again. Every step records in a table what it added, by which rule and from which data.

**Order of inclusion (the three tiers of S2.3)**
1. **Character floor**: every standard character of each language becomes a single token, with no measurement. The check is
   0% byte-fallback tokens on spoken data.
2. **Frequent word forms**: short, frequent spoken forms. The amount is set by the word cut rate (S2.2).
3. **Subwords**: pieces that cut the words tier 2 lacks into fewer tokens; sized by the cut rate where tier 2 does not
   reach the target, small elsewhere.

**Blocks (S12.1a, S12.4)**
- Base block: all 32,000 Helium tokens, plus Qwen3's digits, punctuation, symbols and byte fragments. Always computed.
- Script blocks: Qwen3 tokens with no identical string in Helium, grouped by script.
- Language blocks: the tokens the three tiers add.

**Output**: one result file per step under `.cache/huggingface/runs/vocab/`.

The code stays in this notebook (AGENTS.md 6.1). Run it from the repository root.

In [1]:
import collections
import json
import os
import re
import unicodedata
from dataclasses import dataclass
import urllib.request
from pathlib import Path

os.environ.setdefault("HF_HOME", str(Path(".cache/huggingface").resolve()))
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from IPython.display import Markdown, display
from transformers import AutoTokenizer

from polyplex_research.utils.assemble import load_base_tokenizer

HELIUM = "nvidia/personaplex-7b-v1"
QWEN = "Qwen/Qwen3-8B"
# CLDR release the standard characters are read from. A fixed tag keeps the result reproducible.
CLDR_RELEASE = "48.2.3"
OUT = Path(".cache/huggingface/runs/vocab")


@dataclass(frozen=True)
class Qwen3Language:
    """One language or dialect Qwen3 lists, with its code in each corpus that has it (None when the corpus lacks it)."""

    name: str
    fineweb2: str | None
    wikipedia: str | None
    fleurs: str | None


# The 119 languages and dialects of the Qwen3 release (https://qwenlm.github.io/blog/qwen3/), in its order and grouped by
# its language families. Codes checked against the Hugging Face listings of FineWeb-2, Wikipedia (20231101) and FLEURS.
# A variety without a corpus of its own shares its parent's code, as Dari shares Persian's, or has none.
QWEN3_LANGUAGES = (
    # Indo-European
    Qwen3Language("English", None, "en", "en_us"),
    Qwen3Language("French", "fra_Latn", "fr", "fr_fr"),
    Qwen3Language("Portuguese", "por_Latn", "pt", "pt_br"),
    Qwen3Language("German", "deu_Latn", "de", "de_de"),
    Qwen3Language("Romanian", "ron_Latn", "ro", "ro_ro"),
    Qwen3Language("Swedish", "swe_Latn", "sv", "sv_se"),
    Qwen3Language("Danish", "dan_Latn", "da", "da_dk"),
    Qwen3Language("Bulgarian", "bul_Cyrl", "bg", "bg_bg"),
    Qwen3Language("Russian", "rus_Cyrl", "ru", "ru_ru"),
    Qwen3Language("Czech", "ces_Latn", "cs", "cs_cz"),
    Qwen3Language("Greek", "ell_Grek", "el", "el_gr"),
    Qwen3Language("Ukrainian", "ukr_Cyrl", "uk", "uk_ua"),
    Qwen3Language("Spanish", "spa_Latn", "es", "es_419"),
    Qwen3Language("Dutch", "nld_Latn", "nl", "nl_nl"),
    Qwen3Language("Slovak", "slk_Latn", "sk", "sk_sk"),
    Qwen3Language("Croatian", "hrv_Latn", "hr", "hr_hr"),
    Qwen3Language("Polish", "pol_Latn", "pl", "pl_pl"),
    Qwen3Language("Lithuanian", "lit_Latn", "lt", "lt_lt"),
    Qwen3Language("Norwegian Bokmal", "nob_Latn", "no", "nb_no"),
    Qwen3Language("Norwegian Nynorsk", "nno_Latn", "nn", None),
    Qwen3Language("Persian", "fas_Arab", "fa", "fa_ir"),
    Qwen3Language("Slovenian", "slv_Latn", "sl", "sl_si"),
    Qwen3Language("Gujarati", "guj_Gujr", "gu", "gu_in"),
    Qwen3Language("Latvian", "lvs_Latn", "lv", "lv_lv"),
    Qwen3Language("Italian", "ita_Latn", "it", "it_it"),
    Qwen3Language("Occitan", "oci_Latn", "oc", "oc_fr"),
    Qwen3Language("Nepali", "npi_Deva", "ne", "ne_np"),
    Qwen3Language("Marathi", "mar_Deva", "mr", "mr_in"),
    Qwen3Language("Belarusian", "bel_Cyrl", "be", "be_by"),
    Qwen3Language("Serbian", "srp_Cyrl", "sr", "sr_rs"),
    Qwen3Language("Luxembourgish", "ltz_Latn", "lb", "lb_lu"),
    Qwen3Language("Venetian", "vec_Latn", "vec", None),
    Qwen3Language("Assamese", "asm_Beng", "as", "as_in"),
    Qwen3Language("Welsh", "cym_Latn", "cy", "cy_gb"),
    Qwen3Language("Silesian", "szl_Latn", "szl", None),
    Qwen3Language("Asturian", "ast_Latn", "ast", "ast_es"),
    Qwen3Language("Chhattisgarhi", "hne_Deva", None, None),
    Qwen3Language("Awadhi", "awa_Deva", None, None),
    Qwen3Language("Maithili", "mai_Deva", "mai", None),
    Qwen3Language("Bhojpuri", "bho_Deva", "bh", None),
    Qwen3Language("Sindhi", "snd_Arab", "sd", "sd_in"),
    Qwen3Language("Irish", "gle_Latn", "ga", "ga_ie"),
    Qwen3Language("Faroese", "fao_Latn", "fo", None),
    Qwen3Language("Hindi", "hin_Deva", "hi", "hi_in"),
    Qwen3Language("Punjabi", "pan_Guru", "pa", "pa_in"),
    Qwen3Language("Bengali", "ben_Beng", "bn", "bn_in"),
    Qwen3Language("Oriya", "ory_Orya", "or", "or_in"),
    Qwen3Language("Tajik", "tgk_Cyrl", "tg", "tg_tj"),
    Qwen3Language("Eastern Yiddish", "ydd_Hebr", "yi", None),
    Qwen3Language("Lombard", "lmo_Latn", "lmo", None),
    Qwen3Language("Ligurian", "lij_Latn", "lij", None),
    Qwen3Language("Sicilian", "scn_Latn", "scn", None),
    Qwen3Language("Friulian", "fur_Latn", "fur", None),
    Qwen3Language("Sardinian", "srd_Latn", "sc", None),
    Qwen3Language("Galician", "glg_Latn", "gl", "gl_es"),
    Qwen3Language("Catalan", "cat_Latn", "ca", "ca_es"),
    Qwen3Language("Icelandic", "isl_Latn", "is", "is_is"),
    Qwen3Language("Tosk Albanian", "als_Latn", "sq", None),
    Qwen3Language("Limburgish", "lim_Latn", "li", None),
    Qwen3Language("Dari", "fas_Arab", None, None),
    Qwen3Language("Afrikaans", "afr_Latn", "af", "af_za"),
    Qwen3Language("Macedonian", "mkd_Cyrl", "mk", "mk_mk"),
    Qwen3Language("Sinhala", "sin_Sinh", "si", None),
    Qwen3Language("Urdu", "urd_Arab", "ur", "ur_pk"),
    Qwen3Language("Magahi", "mag_Deva", None, None),
    Qwen3Language("Bosnian", "bos_Latn", "bs", "bs_ba"),
    Qwen3Language("Armenian", "hye_Armn", "hy", "hy_am"),
    # Sino-Tibetan
    Qwen3Language("Simplified Chinese", "cmn_Hani", "zh", "cmn_hans_cn"),
    Qwen3Language("Traditional Chinese", "cmn_Hani", "zh", None),
    Qwen3Language("Cantonese", "yue_Hani", "zh-yue", "yue_hant_hk"),
    Qwen3Language("Burmese", "mya_Mymr", "my", "my_mm"),
    # Afro-Asiatic
    Qwen3Language("Standard Arabic", "arb_Arab", "ar", "ar_eg"),
    Qwen3Language("Najdi Arabic", "ars_Arab", None, None),
    Qwen3Language("Levantine Arabic", "apc_Arab", None, None),
    Qwen3Language("Egyptian Arabic", "arz_Arab", "arz", None),
    Qwen3Language("Moroccan Arabic", "ary_Arab", "ary", None),
    Qwen3Language("Mesopotamian Arabic", "acm_Arab", None, None),
    Qwen3Language("Taizzi-Adeni Arabic", None, None, None),
    Qwen3Language("Tunisian Arabic", "aeb_Arab", None, None),
    Qwen3Language("Hebrew", "heb_Hebr", "he", "he_il"),
    Qwen3Language("Maltese", "mlt_Latn", "mt", "mt_mt"),
    # Austronesian
    Qwen3Language("Indonesian", "ind_Latn", "id", "id_id"),
    Qwen3Language("Malay", "zsm_Latn", "ms", "ms_my"),
    Qwen3Language("Tagalog", "fil_Latn", "tl", "fil_ph"),
    Qwen3Language("Cebuano", "ceb_Latn", "ceb", "ceb_ph"),
    Qwen3Language("Javanese", "jav_Latn", "jv", "jv_id"),
    Qwen3Language("Sundanese", "sun_Latn", "su", None),
    Qwen3Language("Minangkabau", "min_Latn", "min", None),
    Qwen3Language("Balinese", "ban_Latn", "ban", None),
    Qwen3Language("Banjar", "bjn_Latn", "bjn", None),
    Qwen3Language("Pangasinan", "pag_Latn", "pag", None),
    Qwen3Language("Iloko", "ilo_Latn", "ilo", None),
    Qwen3Language("Waray", "war_Latn", "war", None),
    # Dravidian
    Qwen3Language("Tamil", "tam_Taml", "ta", "ta_in"),
    Qwen3Language("Telugu", "tel_Telu", "te", "te_in"),
    Qwen3Language("Kannada", "kan_Knda", "kn", "kn_in"),
    Qwen3Language("Malayalam", "mal_Mlym", "ml", "ml_in"),
    # Turkic
    Qwen3Language("Turkish", "tur_Latn", "tr", "tr_tr"),
    Qwen3Language("North Azerbaijani", "azj_Latn", "az", "az_az"),
    Qwen3Language("Northern Uzbek", "uzn_Latn", "uz", "uz_uz"),
    Qwen3Language("Kazakh", "kaz_Cyrl", "kk", "kk_kz"),
    Qwen3Language("Bashkir", "bak_Cyrl", "ba", None),
    Qwen3Language("Tatar", "tat_Cyrl", "tt", None),
    # Tai-Kadai
    Qwen3Language("Thai", "tha_Thai", "th", "th_th"),
    Qwen3Language("Lao", "lao_Laoo", "lo", "lo_la"),
    # Uralic
    Qwen3Language("Finnish", "fin_Latn", "fi", "fi_fi"),
    Qwen3Language("Estonian", "ekk_Latn", "et", "et_ee"),
    Qwen3Language("Hungarian", "hun_Latn", "hu", "hu_hu"),
    # Austroasiatic
    Qwen3Language("Vietnamese", "vie_Latn", "vi", "vi_vn"),
    Qwen3Language("Khmer", "khm_Khmr", "km", "km_kh"),
    # Other
    Qwen3Language("Japanese", "jpn_Jpan", "ja", "ja_jp"),
    Qwen3Language("Korean", "kor_Hang", "ko", "ko_kr"),
    Qwen3Language("Georgian", "kat_Geor", "ka", "ka_ge"),
    Qwen3Language("Basque", "eus_Latn", "eu", None),
    Qwen3Language("Haitian", "hat_Latn", "ht", None),
    Qwen3Language("Papiamento", "pap_Latn", "pap", None),
    Qwen3Language("Kabuverdianu", "kea_Latn", None, "kea_cv"),
    Qwen3Language("Tok Pisin", "tpi_Latn", "tpi", None),
    Qwen3Language("Swahili", "swh_Latn", "sw", "sw_ke"),
)

OUT.mkdir(parents=True, exist_ok=True)


import time

THREADS = os.cpu_count()
started = time.time()
# Languages are searched independently, each in a forked process with a share of the cores.
WORKERS = max(1, min(32, THREADS // 2))
WORKER_THREADS = max(1, THREADS // WORKERS)


def in_workers(function, items):
    if WORKERS == 1:
        return [function(item) for item in items]
    import multiprocessing

    with multiprocessing.get_context("fork").Pool(WORKERS) as pool:
        return pool.map(function, items, chunksize=1)


def elapsed(label):
    # Also appended to a log file, so a running notebook shows its progress.
    line = f"[{label}] {time.time() - started:,.0f} s since the start"
    print(line, flush=True)
    with open(OUT / "build_vocab_progress.log", "a") as log:
        log.write(f"{time.strftime('%H:%M:%S')} {line}\n")


def table(rows, header):
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

W1024 18:59:12.231000 3827323 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1024 18:59:12.301000 3827323 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


## 1. Matching the base vocabularies

Qwen3 and Helium tokens are matched by surface string. Helium's `▁` (SentencePiece) and Qwen3's `Ġ` (byte BPE) both mean a
leading space. Tokens with the same string are treated as one token and kept as the Helium token (base block, Helium head rows
preserved). Qwen3 tokens missing from Helium are grouped by script into script blocks. Digits, punctuation, symbols and byte
fragments go to the base block.

In [2]:
helium = load_base_tokenizer(HELIUM)
qwen = AutoTokenizer.from_pretrained(QWEN)

helium_surface = {}
helium_special = []
for index in range(len(helium)):
    piece = helium.convert_ids_to_tokens(index)
    if piece.startswith("<") and piece.endswith(">"):
        helium_special.append(piece)
        continue
    helium_surface.setdefault(piece.replace("▁", " "), index)

# Qwen3's added tokens are markers, not text. Its tool-call and thinking markers enter as control tokens; the fill-in-the-middle
# and repository markers are left out (S12.1a). A Qwen3 string equal to a Helium control token (`<s>`) is left out too.
QWEN_CONTROL = ["<tool_call>", "</tool_call>", "<tool_response>", "</tool_response>", "<think>", "</think>"]
qwen_surface = {}
for token, index in qwen.get_vocab().items():
    if index in qwen.all_special_ids or token in qwen.get_added_vocab():
        continue
    surface = qwen.convert_tokens_to_string([token])
    if surface not in helium_special:
        qwen_surface.setdefault(surface, index)

shared = set(helium_surface) & set(qwen_surface)
helium_only = set(helium_surface) - set(qwen_surface)

SCRIPTS = ("LATIN", "CYRILLIC", "ARABIC", "HEBREW", "GREEK", "THAI", "LAO", "KHMER", "MYANMAR", "GEORGIAN", "ARMENIAN",
           "DEVANAGARI", "BENGALI", "GURMUKHI", "GUJARATI", "ORIYA", "TAMIL", "TELUGU", "KANNADA", "MALAYALAM", "SINHALA")


def script_of(ch):
    code = ord(ch)
    if 0xAC00 <= code <= 0xD7A3 or 0x1100 <= code <= 0x11FF or 0x3130 <= code <= 0x318F:
        return "hangul"
    if 0x3040 <= code <= 0x30FF or 0x31F0 <= code <= 0x31FF:
        return "kana"
    if 0x4E00 <= code <= 0x9FFF or 0x3400 <= code <= 0x4DBF or 0xF900 <= code <= 0xFAFF or 0x20000 <= code <= 0x2FFFF:
        return "han"
    if ch.isascii():
        return "latin" if ch.isalpha() else None
    name = unicodedata.name(ch, "")
    for script in SCRIPTS:
        if name.startswith(script):
            return script.lower()
    if unicodedata.category(ch)[0] in "PSZNC":
        return None
    return "other"


def block_of(surface):
    if "�" in surface or not surface.strip():
        return "base"
    scripts = {script_of(ch) for ch in surface} - {None}
    if not scripts:
        return "base"
    if scripts <= {"han", "kana"}:
        return "kana" if "kana" in scripts else "han"
    if len(scripts) == 1:
        return next(iter(scripts))
    return "mixed"


qwen_blocks = collections.defaultdict(list)
for surface, index in qwen_surface.items():
    if surface in shared:
        continue
    qwen_blocks[block_of(surface)].append(surface)

base_size = len(helium) + len(QWEN_CONTROL) + len(qwen_blocks["base"])
table(
    [
        ["Helium vocabulary (with special tokens)", len(helium)],
        ["  of which special tokens", len(helium_special)],
        ["Qwen3 vocabulary (without special and added tokens)", len(qwen_surface)],
        ["Qwen3 tool-call and thinking markers (control tokens)", len(QWEN_CONTROL)],
        ["Tokens with the same string in both (kept as Helium tokens)", len(shared)],
        ["Helium-only tokens", len(helium_only)],
        ["Qwen3-only digits, punctuation, symbols and byte fragments (base block)", len(qwen_blocks["base"])],
        ["**Base block total**", base_size],
    ],
    ["Item", "Count"],
)
table(
    sorted(([name, len(tokens)] for name, tokens in qwen_blocks.items() if name != "base"), key=lambda row: -row[1]),
    ["Script block (Qwen3-only tokens)", "Count"],
)
json.dump({name: sorted(tokens) for name, tokens in qwen_blocks.items()}, open(OUT / "qwen_blocks.json", "w"), ensure_ascii=False)

| Item | Count |
|---|---|
| Helium vocabulary (with special tokens) | 32000 |
|   of which special tokens | 261 |
| Qwen3 vocabulary (without special and added tokens) | 150303 |
| Qwen3 tool-call and thinking markers (control tokens) | 6 |
| Tokens with the same string in both (kept as Helium tokens) | 26471 |
| Helium-only tokens | 5268 |
| Qwen3-only digits, punctuation, symbols and byte fragments (base block) | 8297 |
| **Base block total** | 40303 |

| Script block (Qwen3-only tokens) | Count |
|---|---|
| latin | 68839 |
| han | 25462 |
| arabic | 3992 |
| cyrillic | 3949 |
| hangul | 3564 |
| hebrew | 3164 |
| thai | 2570 |
| kana | 2067 |
| other | 1231 |
| greek | 193 |
| armenian | 77 |
| devanagari | 63 |
| bengali | 44 |
| myanmar | 37 |
| georgian | 37 |
| khmer | 36 |
| lao | 33 |
| malayalam | 32 |
| tamil | 28 |
| sinhala | 26 |
| kannada | 25 |
| mixed | 19 |
| gujarati | 16 |
| telugu | 14 |
| gurmukhi | 10 |
| oriya | 7 |

## 2. Tier 1: the character floor

Each language gets every character of its standard lists, with no measurement (S2.3). Tier 1 comes from the standard lists
only and is never extended from data.

| Applies to | Standard list |
|---|---|
| All languages | CLDR exemplar characters (`exemplarCharacters`), punctuation (`punctuation`), and the digits of the default and native (`native`) numbering systems. Upper case too, where the script has case |
| Indic, Southeast Asian and Georgian scripts | In addition, every assigned character of the script's Unicode block, plus ZWJ and ZWNJ |
| Korean | Hangul from the 2,350 KS X 1001 syllables |
| Japanese | Kanji from JIS X 0208 levels 1 and 2 |
| Simplified Chinese | Hanzi from GB 2312 |
| Traditional Chinese, Cantonese | Hanzi from the 5,401 Big5 common characters (常用字) |

A language missing from CLDR takes its parent language's set, noted in the table. Characters that are already single Qwen3
tokens are not added again. Non-standard characters found in spoken data are kept aside as candidates for tiers 2 and 3.

In [3]:
CLDR_LOCALE = {
    "English": "en", "Norwegian Bokmal": "nb", "Norwegian Nynorsk": "nn", "Persian": "fa", "Dari": "fa-AF",
    "Simplified Chinese": "zh", "Traditional Chinese": "zh-Hant", "Cantonese": "yue",
    "Standard Arabic": "ar", "Najdi Arabic": "ar-SA", "Levantine Arabic": "apc", "Egyptian Arabic": "ar-EG",
    "Moroccan Arabic": "ar-MA", "Mesopotamian Arabic": "ar-IQ", "Taizzi-Adeni Arabic": "ar-YE", "Tunisian Arabic": "ar-TN",
    "Tagalog": "fil", "Malay": "ms", "Serbian": "sr", "Tosk Albanian": "sq", "Eastern Yiddish": "yi",
    "Northern Uzbek": "uz", "North Azerbaijani": "az", "Latvian": "lv", "Estonian": "et", "Nepali": "ne", "Swahili": "sw",
    "Bhojpuri": "bho", "Maithili": "mai", "Kabuverdianu": "kea", "Sardinian": "sc",
}
# Languages CLDR does not describe, read from the closest language it does.
CLDR_PARENT = {
    "Chhattisgarhi": "hi", "Awadhi": "hi", "Magahi": "hi", "Venetian": "it", "Lombard": "it", "Ligurian": "it",
    "Sicilian": "it", "Friulian": "it", "Silesian": "pl", "Limburgish": "nl", "Minangkabau": "id", "Balinese": "id",
    "Banjar": "id", "Pangasinan": "fil", "Iloko": "fil", "Waray": "fil", "Papiamento": "es", "Tok Pisin": "en",
    "Haitian": "fr", "Tatar": "ru", "Bashkir": "ru",
}

cache = Path(".cache/cldr") / CLDR_RELEASE
cache.mkdir(parents=True, exist_ok=True)


def cldr_file(package, locale, name):
    path = cache / package / f"{locale}.{name}.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        url = (f"https://raw.githubusercontent.com/unicode-org/cldr-json/{CLDR_RELEASE}/cldr-json/{package}/"
               + (f"main/{locale}/{name}.json" if locale else f"{name}.json"))
        try:
            with urllib.request.urlopen(url, timeout=60) as response:
                path.write_bytes(response.read())
        except urllib.error.HTTPError:
            return None
    return json.load(open(path))


def cldr_characters(locale):
    data = cldr_file("cldr-misc-full", locale, "characters")
    return None if data is None else data["main"][locale]["characters"]


NUMBERING_SYSTEMS = cldr_file("cldr-core", None, "supplemental/numberingSystems")["supplemental"]["numberingSystems"]


def cldr_digits(locale):
    data = cldr_file("cldr-numbers-full", locale, "numbers")
    if data is None:
        return set()
    numbers = data["main"][locale]["numbers"]
    systems = {numbers["defaultNumberingSystem"], *numbers.get("otherNumberingSystems", {}).values()}
    return {digit for system in systems for digit in NUMBERING_SYSTEMS.get(system, {}).get("_digits", "")}


def expand_exemplars(text):
    # CLDR writes combining marks and some symbols as \uXXXX escapes, and quotes other characters with a backslash.
    text = re.sub(r"\\u([0-9A-Fa-f]{4})", lambda match: chr(int(match.group(1), 16)), text).replace("\\", "")
    items = []
    for part in text.strip("[]").split():
        if part.startswith("{") and part.endswith("}"):
            items.append(part[1:-1])
        elif len(part) == 3 and part[1] == "-":
            items += [chr(code) for code in range(ord(part[0]), ord(part[2]) + 1)]
        else:
            items += list(part)
    with_case = set(items)
    with_case |= {item.upper() for item in items if item.upper() != item}
    return with_case


standard = {}
source = {}
for language in QWEN3_LANGUAGES:
    if language.name == "Korean":
        continue
    candidates = [CLDR_LOCALE.get(language.name), language.wikipedia, CLDR_PARENT.get(language.name)]
    for locale in [code for code in candidates if code]:
        characters = cldr_characters(locale)
        if characters is not None:
            standard[language.name] = (expand_exemplars(characters["exemplarCharacters"])
                                       | expand_exemplars(characters.get("punctuation", ""))
                                       | cldr_digits(locale))
            source[language.name] = locale if locale != CLDR_PARENT.get(language.name) else f"{locale} (related language)"
            break
missing_cldr = [language.name for language in QWEN3_LANGUAGES if language.name not in standard and language.name != "Korean"]
print("Languages with no CLDR exemplar characters:", missing_cldr)

Languages with no CLDR exemplar characters: []


### Korean: KS X 1001

The Korean spoken data is the v1 data: every utterance of the AI Hub daily conversation multi-session corpus (011) and the
Korean source text of the AI Hub colloquial Korean-English corpus.

In [4]:
KOREAN_SPOKEN = Path(".cache/datasets")


def korean_spoken_texts():
    for path in sorted((KOREAN_SPOKEN / "multisession_daily").rglob("*.json")):
        for session in json.load(open(path))["sessionInfo"]:
            for turn in session["dialog"]:
                yield turn["utterance"]
    for path in sorted((KOREAN_SPOKEN / "colloquial_koen").rglob("*한영*.json")):
        data = json.load(open(path))
        for row in data["data"] if isinstance(data, dict) else data:
            yield row["ko_original"]


korean_texts = list(korean_spoken_texts())
ks_x_1001 = {chr(code) for code in range(0xAC00, 0xD7A4) if len(chr(code).encode("euc-kr", "ignore")) == 2}
syllables_seen = collections.Counter(ch for text in korean_texts for ch in text if 0xAC00 <= ord(ch) <= 0xD7A3)
outside = {ch: count for ch, count in syllables_seen.items() if ch not in ks_x_1001}
korean_cldr = cldr_characters("ko")
standard["Korean"] = ks_x_1001 | expand_exemplars(korean_cldr["punctuation"]) | cldr_digits("ko")
# Syllables the spoken data uses outside KS X 1001 are candidates for tiers 2 and 3, not tier 1.
json.dump(outside, open(OUT / "korean_candidates_outside_ks.json", "w"), ensure_ascii=False, indent=1)
source["Korean"] = "KS X 1001 + CLDR punctuation and digits"
print(f"{len(korean_texts):,} spoken sentences, {len(syllables_seen):,} distinct syllables, {len(outside)} of them outside KS X 1001")
print("Syllables outside KS X 1001 kept as tier 2 and 3 candidates, most frequent first:", sorted(outside.items(), key=lambda kv: -kv[1])[:30])

3,273,481 spoken sentences, 2,501 distinct syllables, 424 of them outside KS X 1001
Syllables outside KS X 1001 kept as tier 2 and 3 candidates, most frequent first: [('됬', 1500), ('몐', 390), ('떄', 185), ('슌', 142), ('귬', 111), ('녓', 61), ('춉', 55), ('웻', 52), ('뱡', 46), ('먄', 43), ('곘', 41), ('똠', 40), ('븡', 37), ('렜', 36), ('픙', 36), ('녤', 34), ('댠', 33), ('됭', 33), ('퀜', 30), ('똣', 26), ('헀', 24), ('닏', 22), ('솁', 22), ('쎼', 21), ('늗', 21), ('럅', 20), ('뢱', 20), ('잌', 19), ('탼', 19), ('졋', 18)]


### Tier 1 tokens per language

A language's tier 1 tokens are its standard characters that are not single tokens in the Qwen3 vocabulary, and the word-start
form (`▁` plus the character, S12.1b) of every standard letter and digit that has no space-led form in Qwen3 or Helium.
Without that form a word starting with the character spends a frame on `▁` alone. Word-start forms are written here with a
leading space, as the other surface strings are.

In [5]:
# Scripts whose CLDR lists leave out vowel signs, letters with a dot or old letters, so the whole Unicode block is taken.
BLOCK_SCRIPTS = {
    "devanagari": (0x0900, 0x097F), "bengali": (0x0980, 0x09FF), "gurmukhi": (0x0A00, 0x0A7F), "gujarati": (0x0A80, 0x0AFF),
    "oriya": (0x0B00, 0x0B7F), "tamil": (0x0B80, 0x0BFF), "telugu": (0x0C00, 0x0C7F), "kannada": (0x0C80, 0x0CFF),
    "malayalam": (0x0D00, 0x0D7F), "sinhala": (0x0D80, 0x0DFF), "thai": (0x0E00, 0x0E7F), "lao": (0x0E80, 0x0EFF),
    "myanmar": (0x1000, 0x109F), "georgian": (0x10A0, 0x10FF), "khmer": (0x1780, 0x17FF),
}


# Zero width joiner and non-joiner, which these scripts' orthographies use between letters.
JOINERS = {"\u200c", "\u200d"}


def block_letters(chars):
    letters = set()
    for script in {script_of(ch) for ch in chars if len(ch) == 1 and ch.isalpha()} & set(BLOCK_SCRIPTS):
        start, end = BLOCK_SCRIPTS[script]
        letters |= {chr(code) for code in range(start, end + 1) if unicodedata.name(chr(code), "")}
        letters |= JOINERS
    return letters


def encoding_hanzi(codec, keep=lambda encoded: True):
    hanzi = set()
    for code in list(range(0x4E00, 0xA000)) + list(range(0x3400, 0x4DC0)) + list(range(0xF900, 0xFB00)):
        try:
            encoded = chr(code).encode(codec)
        except UnicodeEncodeError:
            continue
        if keep(encoded):
            hanzi.add(chr(code))
    return hanzi


def big5_common(encoded):
    # Big5 level 1 (常用字) runs from A440 to C67E.
    return len(encoded) == 2 and (0xA4 <= encoded[0] < 0xC6 or (encoded[0] == 0xC6 and encoded[1] <= 0x7E))


HANZI_STANDARD = {
    "Japanese": ("JIS X 0208", encoding_hanzi("euc_jp", lambda encoded: len(encoded) == 2)),
    "Simplified Chinese": ("GB 2312", encoding_hanzi("gb2312")),
    "Traditional Chinese": ("Big5 common characters", encoding_hanzi("big5", big5_common)),
    "Cantonese": ("Big5 common characters", encoding_hanzi("big5", big5_common)),
}
for name, (label, hanzi) in HANZI_STANDARD.items():
    # The encoding standard replaces CLDR's ideographs; kana, punctuation and digits stay as CLDR lists them.
    standard[name] = {ch for ch in standard[name] if script_of(ch) != "han"} | hanzi
    source[name] = f"{source[name]}, Han characters from {label}"
for language in QWEN3_LANGUAGES:
    extra = block_letters(standard.get(language.name, set()))
    if extra - standard.get(language.name, set()):
        standard[language.name] = standard[language.name] | extra
        source[language.name] = f"{source[language.name]} + Unicode block"

spaced = set(qwen_surface) | set(helium_surface)
tier1 = {}
rows = []
for language in QWEN3_LANGUAGES:
    chars = standard.get(language.name, set())
    added = sorted(ch for ch in chars if ch not in qwen_surface)
    starts = sorted(" " + ch for ch in chars if len(ch) == 1 and (ch.isalpha() or ch.isdigit()) and " " + ch not in spaced)
    tier1[language.name] = added + starts
    rows.append([language.name, source.get(language.name, "none"), len(chars), len(chars) - len(added), len(added), len(starts)])
table(rows, ["Language", "Standard source", "Standard characters", "Already in Qwen3", "Added as tier 1", "Word-start forms added"])
all_tier1 = sorted({ch for chars in tier1.values() for ch in chars})
print(f"Tier 1 tokens summed over languages: {sum(len(v) for v in tier1.values()):,}, distinct: {len(all_tier1):,}")
json.dump({"cldr_release": CLDR_RELEASE, "source": source, "tier1": tier1}, open(OUT / "tier1.json", "w"), ensure_ascii=False, indent=1)

| Language | Standard source | Standard characters | Already in Qwen3 | Added as tier 1 | Word-start forms added |
|---|---|---|---|---|---|
| English | en | 94 | 94 | 0 | 10 |
| French | fr | 124 | 124 | 0 | 23 |
| Portuguese | pt | 120 | 120 | 0 | 19 |
| German | de | 102 | 102 | 0 | 11 |
| Romanian | ro | 99 | 99 | 0 | 16 |
| Swedish | sv | 104 | 104 | 0 | 10 |
| Danish | da | 98 | 98 | 0 | 11 |
| Bulgarian | bg | 98 | 98 | 0 | 19 |
| Russian | ru | 108 | 108 | 0 | 23 |
| Czech | cs | 118 | 118 | 0 | 28 |
| Greek | el | 104 | 101 | 3 | 50 |
| Ukrainian | uk | 104 | 104 | 0 | 25 |
| Spanish | es | 112 | 112 | 0 | 15 |
| Dutch | nl | 116 | 114 | 2 | 17 |
| Slovak | sk | 126 | 123 | 3 | 28 |
| Croatian | hr | 97 | 94 | 3 | 14 |
| Polish | pl | 111 | 111 | 0 | 21 |
| Lithuanian | lt | 94 | 93 | 1 | 24 |
| Norwegian Bokmal | nb | 110 | 110 | 0 | 14 |
| Norwegian Nynorsk | nn | 110 | 110 | 0 | 14 |
| Persian | fa | 75 | 62 | 13 | 19 |
| Slovenian | sl | 84 | 84 | 0 | 12 |
| Gujarati | gu + Unicode block | 135 | 59 | 76 | 73 |
| Latvian | lv | 110 | 110 | 0 | 28 |
| Italian | it | 102 | 102 | 0 | 16 |
| Occitan | oc | 115 | 115 | 0 | 17 |
| Nepali | ne + Unicode block | 149 | 81 | 68 | 86 |
| Marathi | mr + Unicode block | 161 | 91 | 70 | 86 |
| Belarusian | be | 94 | 91 | 3 | 22 |
| Serbian | sr | 93 | 93 | 0 | 22 |
| Luxembourgish | lb | 100 | 100 | 0 | 12 |
| Venetian | vec | 108 | 107 | 1 | 17 |
| Assamese | as + Unicode block | 134 | 80 | 54 | 65 |
| Welsh | cy | 156 | 153 | 3 | 41 |
| Silesian | szl | 115 | 115 | 0 | 23 |
| Asturian | ast | 110 | 109 | 1 | 19 |
| Chhattisgarhi | hi (related language) + Unicode block | 171 | 103 | 68 | 96 |
| Awadhi | hi (related language) + Unicode block | 171 | 103 | 68 | 96 |
| Maithili | mai + Unicode block | 180 | 108 | 72 | 96 |
| Bhojpuri | bho + Unicode block | 172 | 104 | 68 | 96 |
| Sindhi | sd | 82 | 64 | 18 | 33 |
| Irish | ga | 88 | 88 | 0 | 13 |
| Faroese | fo | 98 | 98 | 0 | 17 |
| Hindi | hi + Unicode block | 171 | 103 | 68 | 96 |
| Punjabi | pa + Unicode block | 119 | 48 | 71 | 71 |
| Bengali | bn + Unicode block | 134 | 80 | 54 | 65 |
| Oriya | or + Unicode block | 139 | 53 | 86 | 73 |
| Tajik | tg | 113 | 110 | 3 | 29 |
| Eastern Yiddish | yi | 74 | 61 | 13 | 16 |
| Lombard | lmo | 87 | 87 | 0 | 10 |
| Ligurian | lij | 132 | 132 | 0 | 27 |
| Sicilian | scn | 108 | 108 | 0 | 21 |
| Friulian | fur | 99 | 99 | 0 | 19 |
| Sardinian | sc | 92 | 92 | 0 | 16 |
| Galician | gl | 114 | 114 | 0 | 17 |
| Catalan | ca | 119 | 119 | 0 | 17 |
| Icelandic | is | 106 | 106 | 0 | 18 |
| Tosk Albanian | sq | 115 | 111 | 4 | 12 |
| Limburgish | nl (related language) | 116 | 114 | 2 | 17 |
| Dari | fa-AF | 75 | 62 | 13 | 19 |
| Afrikaans | af | 116 | 116 | 0 | 17 |
| Macedonian | mk | 95 | 95 | 0 | 24 |
| Sinhala | si + Unicode block | 135 | 69 | 66 | 79 |
| Urdu | ur | 70 | 60 | 10 | 32 |
| Magahi | hi (related language) + Unicode block | 171 | 103 | 68 | 96 |
| Bosnian | bs | 96 | 93 | 3 | 14 |
| Armenian | hy | 98 | 90 | 8 | 86 |
| Simplified Chinese | zh, Han characters from GB 2312 | 6828 | 6705 | 123 | 6727 |
| Traditional Chinese | zh-Hant, Han characters from Big5 common characters | 5543 | 4119 | 1424 | 5377 |
| Cantonese | yue, Han characters from Big5 common characters | 5543 | 4119 | 1424 | 5377 |
| Burmese | my + Unicode block | 182 | 58 | 124 | 94 |
| Standard Arabic | ar | 84 | 71 | 13 | 26 |
| Najdi Arabic | ar-SA | 74 | 61 | 13 | 16 |
| Levantine Arabic | apc | 76 | 73 | 3 | 16 |
| Egyptian Arabic | ar-EG | 74 | 61 | 13 | 16 |
| Moroccan Arabic | ar-MA | 84 | 71 | 13 | 26 |
| Mesopotamian Arabic | ar-IQ | 74 | 61 | 13 | 16 |
| Taizzi-Adeni Arabic | ar-YE | 74 | 61 | 13 | 16 |
| Tunisian Arabic | ar-TN | 84 | 71 | 13 | 26 |
| Hebrew | he | 58 | 58 | 0 | 15 |
| Maltese | mt | 98 | 96 | 2 | 22 |
| Indonesian | id | 94 | 94 | 0 | 10 |
| Malay | ms | 94 | 94 | 0 | 10 |
| Tagalog | fil | 95 | 95 | 0 | 12 |
| Cebuano | ceb | 74 | 74 | 0 | 10 |
| Javanese | jv | 93 | 83 | 10 | 27 |
| Sundanese | su | 96 | 96 | 0 | 10 |
| Minangkabau | id (related language) | 94 | 94 | 0 | 10 |
| Balinese | id (related language) | 94 | 94 | 0 | 10 |
| Banjar | id (related language) | 94 | 94 | 0 | 10 |
| Pangasinan | fil (related language) | 95 | 95 | 0 | 12 |
| Iloko | fil (related language) | 95 | 95 | 0 | 12 |
| Waray | fil (related language) | 95 | 95 | 0 | 12 |
| Tamil | ta + Unicode block | 116 | 71 | 45 | 57 |
| Telugu | te + Unicode block | 134 | 47 | 87 | 77 |
| Kannada | kn + Unicode block | 135 | 68 | 67 | 77 |
| Malayalam | ml + Unicode block | 165 | 77 | 88 | 87 |
| Turkish | tr | 100 | 100 | 0 | 13 |
| North Azerbaijani | az | 106 | 106 | 0 | 15 |
| Northern Uzbek | uz | 99 | 95 | 4 | 11 |
| Kazakh | kk | 126 | 126 | 0 | 40 |
| Bashkir | ba | 126 | 125 | 1 | 41 |
| Tatar | tt | 118 | 117 | 1 | 35 |
| Thai | th + Unicode block | 125 | 110 | 15 | 69 |
| Lao | lo + Unicode block | 127 | 76 | 51 | 76 |
| Finnish | fi | 96 | 96 | 0 | 11 |
| Estonian | et | 92 | 92 | 0 | 13 |
| Hungarian | hu | 141 | 120 | 21 | 17 |
| Vietnamese | vi | 220 | 220 | 0 | 120 |
| Khmer | km + Unicode block | 145 | 64 | 81 | 74 |
| Japanese | ja, Han characters from JIS X 0208 | 6627 | 4311 | 2316 | 6501 |
| Korean | KS X 1001 + CLDR punctuation and digits | 2443 | 2165 | 278 | 2154 |
| Georgian | ka + Unicode block | 169 | 84 | 85 | 130 |
| Basque | eu | 98 | 98 | 0 | 12 |
| Haitian | ht | 101 | 101 | 0 | 12 |
| Papiamento | pap | 99 | 99 | 0 | 16 |
| Kabuverdianu | kea | 98 | 97 | 1 | 12 |
| Tok Pisin | tpi | 67 | 67 | 0 | 10 |
| Swahili | sw | 74 | 74 | 0 | 10 |

Tier 1 tokens summed over languages: 37,299, distinct: 18,372


### Check: do byte fragments drop to 0% on spoken Korean?

Spoken sentences are cut with the unchanged Qwen3 tokenizer, and each byte-fragment token (a token that is not a character on
its own) is traced to the character it came from. Fragments are split by position: the first character of a word (after a
space) or a character inside a word. Any fragment tier 1 does not cover is examined before deciding.

In [6]:
covered = set(qwen_surface) | set(tier1["Korean"])
fragment_tokens = total_tokens = 0
causes = collections.Counter()
for text in korean_texts[:20000]:
    encoding = qwen(text, add_special_tokens=False, return_offsets_mapping=True)
    total_tokens += len(encoding["input_ids"])
    for token_id, (start, end) in zip(encoding["input_ids"], encoding["offset_mapping"]):
        if "�" not in qwen.decode([token_id]):
            continue
        fragment_tokens += 1
        position = start
        while position < len(text) and text[position] == " ":
            position += 1
        if position < len(text):
            causes[(text[position], position > 0 and text[position - 1] == " ")] += 1
print(f"Qwen3 as is: byte-fragment tokens {fragment_tokens / total_tokens:.1%} ({fragment_tokens:,}/{total_tokens:,}), 20,000 sentences")
rows = []
for label, word_start in (("first character of a word", True), ("character inside a word", False)):
    forms = {ch: count for (ch, start), count in causes.items() if start == word_start}
    in_qwen = sum(count for ch, count in forms.items() if ch in qwen_surface)
    in_tier1 = sum(count for ch, count in forms.items() if ch not in qwen_surface and ch in covered)
    rest = sum(count for ch, count in forms.items() if ch not in covered)
    rows.append([label, sum(forms.values()), in_qwen, in_tier1, rest])
table(rows, ["Fragment position", "Fragments", "Character is a single Qwen3 token", "Character is in tier 1", "Not covered"])

Qwen3 as is: byte-fragment tokens 14.5% (84,206/580,161), 20,000 sentences


| Fragment position | Fragments | Character is a single Qwen3 token | Character is in tier 1 | Not covered |
|---|---|---|---|---|
| first character of a word | 80663 | 80626 | 4 | 33 |
| character inside a word | 3543 | 3358 | 14 | 171 |

## 3. Tokenizer comparison: byte BPE and Unigram with byte fallback

The same token list is used to cut text in two ways (S12.1b).

- **Qwen3 byte BPE**: the Qwen3 tokenizer as is. Each word is cut with a leading space, as the duplex stream cuts words.
- **Unigram**: Qwen3 and Helium token strings, the leading space written as `▁` (Qwen3's `Ġ` becomes `▁`, S12.1b). A string
  both have is one token, Helium's. `▁x` and `x` stay two tokens. Byte-fragment tokens are dropped and replaced by
  `byte_fallback` (256 byte tokens). Every word is cut with `▁` in front (`add_dummy_prefix`), as `text_stream` cuts a word
  with a leading space, so the token list marks every word start whether or not an `EPAD` precedes it.
- **Unigram + tier 1**: the above plus the tier 1 tokens of every language (characters and word-start forms).

The processor turns input text into NFC before cutting it (S12.1), so every token enters as its NFC form and tokens that become
the same are merged. Vietnamese `ấ` written as a base letter plus a combining mark joins the composed `ấ`, and Devanagari `क़`,
which NFC splits into `क` and a nukta, stays one token as that two-character string. Helium tokens keep their form for their ids.

Unigram scores come from EM: cut the training corpus into its most likely segmentation (Viterbi) with the fixed token list,
re-score each token by how often it was used, and repeat. The training corpus is spoken training data (the 011 training split
and the Korean and English of the colloquial Korean-English training split). The comparison uses 3,000 Korean and 3,000
English sentences from the colloquial Korean-English validation split, which does not overlap the training data.

In [7]:
import math
import random

import sentencepiece as spm
from sentencepiece import sentencepiece_model_pb2 as model_pb2


def colloquial(split_dir, field, limit=None):
    texts = []
    for path in sorted((KOREAN_SPOKEN / "colloquial_koen").glob(f"{split_dir}/*한영*.json")):
        data = json.load(open(path))
        texts += [row[field] for row in (data["data"] if isinstance(data, dict) else data)]
    return texts[:limit] if limit else texts


def multisession_training():
    for path in sorted((KOREAN_SPOKEN / "multisession_daily" / "Training").rglob("*.json")):
        for session in json.load(open(path))["sessionInfo"]:
            for turn in session["dialog"]:
                yield turn["utterance"]


eval_ko = colloquial("VL1", "ko_original", 3000)
eval_en = colloquial("VL1", "en", 3000)
random.seed(0)
train_ko = random.sample(list(multisession_training()), 150_000) + random.sample(colloquial("TL2", "ko_original"), 150_000)
train_en = random.sample(colloquial("TL2", "en"), 300_000)
train_words = [word for text in train_ko + train_en for word in text.split()]
print(f"EM training words: {len(train_words):,}, evaluation sentences: Korean {len(eval_ko):,}, English {len(eval_en):,}")

EM training words: 5,936,282, evaluation sentences: Korean 3,000, English 3,000


In [8]:
def as_piece(surface):
    # A leading space is SentencePiece's word-start mark.
    return "▁" + surface[1:] if surface.startswith(" ") else surface


def unigram_pieces(extra=()):
    # Qwen3 and Helium strings as pieces; a string both have is one piece.
    pieces = set()
    for surface in list(qwen_surface) + list(helium_surface) + list(extra):
        if "�" in surface:
            continue
        piece = as_piece(surface)
        # Input text is NFC, so a piece enters as its NFC form; Helium's are kept as they are for their ids.
        if surface not in helium_surface:
            piece = unicodedata.normalize("NFC", piece)
        # SentencePiece takes no whitespace or control character inside a piece.
        if piece and not any(ch.isspace() or unicodedata.category(ch) == "Cc" for ch in piece):
            pieces.add(piece)
    return sorted(pieces)


def build_unigram(pieces, scores):
    model = model_pb2.ModelProto()
    model.trainer_spec.model_type = model_pb2.TrainerSpec.UNIGRAM
    model.trainer_spec.byte_fallback = True
    model.trainer_spec.unk_id, model.trainer_spec.bos_id, model.trainer_spec.eos_id, model.trainer_spec.pad_id = 0, 1, 2, -1
    model.normalizer_spec.name = "identity"
    model.normalizer_spec.add_dummy_prefix = True
    model.normalizer_spec.remove_extra_whitespaces = False
    model.normalizer_spec.escape_whitespaces = True
    for piece, kind in (("<unk>", model_pb2.ModelProto.SentencePiece.UNKNOWN), ("<s>", model_pb2.ModelProto.SentencePiece.CONTROL),
                        ("</s>", model_pb2.ModelProto.SentencePiece.CONTROL)):
        model.pieces.add(piece=piece, score=0.0, type=kind)
    for byte in range(256):
        model.pieces.add(piece=f"<0x{byte:02X}>", score=0.0, type=model_pb2.ModelProto.SentencePiece.BYTE)
    for piece in pieces:
        model.pieces.add(piece=piece, score=scores[piece], type=model_pb2.ModelProto.SentencePiece.NORMAL)
    return spm.SentencePieceProcessor(model_proto=model.SerializeToString())


def fit_unigram(pieces, words, iterations=4, verbose=True):
    # Hard EM on a fixed vocabulary: cut with the current scores, then score each piece by how often it was used.
    # `words` is a list of words or a Counter of them.
    scores = {piece: -math.log(len(pieces)) for piece in pieces}
    word_counts = words if isinstance(words, collections.Counter) else collections.Counter(words)
    for iteration in range(iterations):
        model = build_unigram(pieces, scores)
        counts = collections.Counter()
        words_in_order = list(word_counts)
        for word, pieces_of_word in zip(words_in_order, model.encode(words_in_order, out_type=str, num_threads=THREADS)):
            count = word_counts[word]
            for piece in pieces_of_word:
                counts[piece] += count
        total = sum(counts[piece] for piece in pieces) + len(pieces) * 0.1
        scores = {piece: math.log((counts[piece] + 0.1) / total) for piece in pieces}
        used = sum(1 for piece in pieces if counts[piece])
        if verbose:
            print(f"  EM {iteration + 1}: tokens used {used:,}/{len(pieces):,}")
    return build_unigram(pieces, scores)


def measure_unigram(model, texts):
    tokens = fragments = words = 0
    for text in texts:
        for word in text.split():
            pieces = model.encode(word, out_type=str)
            words += 1
            tokens += len(pieces)
            fragments += sum(1 for piece in pieces if piece.startswith("<0x"))
    return tokens / words, fragments / tokens


def measure_bpe(texts):
    tokens = fragments = words = 0
    for text in texts:
        for word in text.split():
            ids = qwen.encode(" " + word, add_special_tokens=False)
            words += 1
            tokens += len(ids)
            fragments += sum(1 for i in ids if "�" in qwen.decode([i]))
    return tokens / words, fragments / tokens


base_pieces = unigram_pieces()
tier1_all = sorted({ch for chars in tier1.values() for ch in chars})
print(f"Unigram tokens: {len(base_pieces):,} from Qwen3 and Helium ({len(qwen_surface):,} and {len(helium_surface):,} strings), tier 1: {len(tier1_all):,}")
print("Unigram EM on Qwen3 and Helium tokens")
unigram_qwen = fit_unigram(base_pieces, train_words)
print("Unigram EM on Qwen3 and Helium tokens + tier 1")
unigram_tier1 = fit_unigram(unigram_pieces(tier1_all), train_words)

rows = []
for name, measure in (("Qwen3 byte BPE", measure_bpe), ("Unigram (Qwen3 and Helium tokens)", lambda t: measure_unigram(unigram_qwen, t)),
                      ("Unigram (Qwen3 and Helium tokens + tier 1)", lambda t: measure_unigram(unigram_tier1, t))):
    ko_tokens, ko_fragments = measure(eval_ko)
    en_tokens, en_fragments = measure(eval_en)
    rows.append([name, f"{ko_tokens:.2f}", f"{ko_fragments:.2%}", f"{en_tokens:.2f}", f"{en_fragments:.2%}"])
table(rows, ["Tokenizer", "Korean tokens per word", "Korean byte fragments", "English tokens per word", "English byte fragments"])
elapsed("tokenizer comparison")


Unigram tokens: 151,634 from Qwen3 and Helium (150,303 and 31,739 strings), tier 1: 18,372
Unigram EM on Qwen3 and Helium tokens


  EM 1: tokens used 45,284/151,634


  EM 2: tokens used 43,313/151,634


  EM 3: tokens used 43,079/151,634


  EM 4: tokens used 43,016/151,634


Unigram EM on Qwen3 and Helium tokens + tier 1


... (2 outputs left out)

  EM 3: tokens used 44,843/169,970


  EM 4: tokens used 44,782/169,970


| Tokenizer | Korean tokens per word | Korean byte fragments | English tokens per word | English byte fragments |
|---|---|---|---|---|
| Qwen3 byte BPE | 2.76 | 15.70% | 1.26 | 0.00% |
| Unigram (Qwen3 and Helium tokens) | 2.70 | 0.00% | 1.26 | 0.00% |
| Unigram (Qwen3 and Helium tokens + tier 1) | 2.51 | 0.00% | 1.25 | 0.00% |

[tokenizer comparison] 254 s since the start


## 4. Spoken measurement per language (Unigram with tier 1)

Each language uses whatever spoken data exists (S12.6): FLEURS transcripts (with utterance durations) and OpenSubtitles
(spoken subtitles, up to 200,000 lines streamed from the start of the language file; the tier 2 candidates come from them). Only languages with neither fall back to
Wikipedia sentences, and the table says so. Lines with control or private-use characters are broken data and are dropped.

Measured values:
- **Byte fragments**: the share of byte tokens. Tier 1 holds the language's characters, so this should be 0% (S2.3).
- **Tokens per word**: tokens per whitespace-separated word. For languages written without spaces (Chinese, Japanese, Thai
  and others) a whitespace unit is not a word, so tokens per character is shown as well.
- **Tokens per second, frame occupancy** (languages with FLEURS): utterance tokens divided by utterance duration (samples /
  16,000). Occupancy is (tokens + one `EPAD` per word) / (duration × 12.5 frames). Above 1, the utterance does not fit its
  frames. This approximates over whole utterances because word timings are missing; the per-word cut rate (S2.2) is
  measured separately on data with word timings.

The Unigram scores here were fitted on Korean and English only (section 3), so tokens of other languages all sit at the floor
score. Their tokens per word can therefore come out higher than with the final scores. The final scores are refitted on a mix
of spoken data from every language (S12.1).

In [9]:
import gzip
import io

from huggingface_hub import hf_hub_download

OPENSUBTITLES = "https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2018/mono/{code}.txt.gz"
OPUS_CODE = {"Simplified Chinese": "zh_cn", "Traditional Chinese": "zh_tw", "Portuguese": "pt_br", "Norwegian Bokmal": "no",
             "Tagalog": "tl", "Malay": "ms"}
spoken_cache = OUT / "spoken"
spoken_cache.mkdir(exist_ok=True)


def fleurs_utterances(code):
    rows = []
    for split in ("train", "dev", "test"):
        path = hf_hub_download("google/fleurs", f"data/{code}/{split}.tsv", repo_type="dataset")
        for line in open(path, encoding="utf-8"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) >= 6:
                rows.append((fields[2], int(fields[5]) / 16000))
    return rows


OPENSUBTITLES_LINES = 200_000


def opensubtitles_lines(code, limit=OPENSUBTITLES_LINES):
    path = spoken_cache / f"opensubtitles_{code}_{limit}.txt"
    if not path.exists():
        lines = []
        try:
            with urllib.request.urlopen(OPENSUBTITLES.format(code=code), timeout=120) as response:
                with gzip.open(response, "rt", encoding="utf-8", errors="replace") as stream:
                    for line in stream:
                        line = line.strip()
                        if line:
                            lines.append(line)
                        if len(lines) >= limit:
                            break
        except urllib.error.HTTPError:
            lines = []
        path.write_text("\n".join(lines), encoding="utf-8")
    text = path.read_text(encoding="utf-8")
    return text.splitlines() if text else []


def wikipedia_sentences(code, limit=30_000):
    import pyarrow.parquet as pq
    from huggingface_hub import HfApi

    files = sorted(f.path for f in HfApi().list_repo_tree("wikimedia/wikipedia", path_in_repo=f"20231101.{code}", repo_type="dataset"))
    path = hf_hub_download("wikimedia/wikipedia", files[0], repo_type="dataset")
    sentences = []
    for batch in pq.ParquetFile(path).iter_batches(columns=["text"]):
        for text in batch.column("text").to_pylist():
            sentences += [line.strip() for line in text.split("\n") if len(line.strip()) > 20]
            if len(sentences) >= limit:
                return sentences[:limit]
    return sentences


def measure_texts(texts):
    tokens = fragments = words = characters = 0
    all_words = [word for text in texts for word in text.split()]
    for word, pieces in zip(all_words, unigram_tier1.encode(all_words, out_type=str, num_threads=THREADS)):
        words += 1
        characters += len(word)
        tokens += len(pieces)
        fragments += sum(1 for piece in pieces if piece.startswith("<0x"))
    return {"tokens": tokens, "words": words, "byte_fragments": fragments / max(tokens, 1),
            "tokens_per_word": tokens / max(words, 1), "tokens_per_character": tokens / max(characters, 1)}


def measure_timing(utterances):
    seconds = tokens = epads = 0
    overflow = 0
    for text, duration in utterances:
        words = text.split()
        count = sum(len(unigram_tier1.encode(word)) for word in words)
        seconds += duration
        tokens += count
        epads += len(words)
        overflow += (count + len(words)) > duration * 12.5
    return {"tokens_per_second": tokens / seconds, "occupancy": (tokens + epads) / (seconds * 12.5),
            "overflow_utterances": overflow / max(len(utterances), 1)}


measurements = {}
spoken = {}
for language in QWEN3_LANGUAGES:
    texts, sources, timing, durations = [], [], None, {}
    if language.fleurs:
        utterances = fleurs_utterances(language.fleurs)
        texts += [text for text, _ in utterances]
        sources.append(f"FLEURS {len(utterances):,}")
        timing = measure_timing(utterances)
        durations = dict(utterances)
    opus = OPUS_CODE.get(language.name, language.wikipedia)
    if opus:
        lines = opensubtitles_lines(opus)
        if lines:
            texts += lines
            sources.append(f"OpenSubtitles {len(lines):,}")
    if not texts and language.wikipedia:
        sentences = wikipedia_sentences(language.wikipedia)
        texts += sentences
        sources.append(f"Wikipedia {len(sentences):,} (not spoken)")
    # Lines carrying control or private-use characters are broken data, not language.
    texts = [text for text in texts if not any(unicodedata.category(ch) in ("Cc", "Co") for ch in text)]
    result = measure_texts(texts) if texts else {}
    spoken[language.name] = {"texts": texts, "durations": durations}
    result["sources"] = ", ".join(sources) or "none"
    if timing:
        result.update(timing)
    measurements[language.name] = result

json.dump(measurements, open(OUT / "spoken_measurements.json", "w"), ensure_ascii=False, indent=1)
rows = []
for language in QWEN3_LANGUAGES:
    m = measurements[language.name]
    rows.append([
        language.name, m["sources"],
        f"{m['byte_fragments']:.2%}" if "byte_fragments" in m else "-",
        f"{m['tokens_per_word']:.2f}" if "tokens_per_word" in m else "-",
        f"{m['tokens_per_character']:.2f}" if "tokens_per_character" in m else "-",
        f"{m['tokens_per_second']:.2f}" if "tokens_per_second" in m else "-",
        f"{m['occupancy']:.2f}" if "occupancy" in m else "-",
        f"{m['overflow_utterances']:.1%}" if "overflow_utterances" in m else "-",
    ])
table(rows, ["Language", "Data", "Byte fragments", "Tokens per word", "Tokens per character", "Tokens per second", "Frame occupancy", "Overflowing utterances"])
elapsed("spoken measurement")


| Language | Data | Byte fragments | Tokens per word | Tokens per character | Tokens per second | Frame occupancy | Overflowing utterances |
|---|---|---|---|---|---|---|---|
| English | FLEURS 3,643, OpenSubtitles 200,000 | 0.00% | 1.34 | 0.31 | 2.57 | 0.37 | 0.0% |
| French | FLEURS 4,158, OpenSubtitles 200,000 | 0.00% | 1.75 | 0.38 | 3.60 | 0.46 | 0.0% |
| Portuguese | FLEURS 4,098, OpenSubtitles 200,000 | 0.00% | 1.80 | 0.38 | 2.98 | 0.38 | 0.0% |
| German | FLEURS 4,212, OpenSubtitles 200,000 | 0.00% | 1.78 | 0.36 | 3.51 | 0.43 | 0.0% |
| Romanian | FLEURS 4,161, OpenSubtitles 200,000 | 0.00% | 2.10 | 0.46 | 4.03 | 0.47 | 0.0% |
| Swedish | FLEURS 3,474, OpenSubtitles 200,000 | 0.00% | 1.91 | 0.44 | 3.46 | 0.41 | 0.0% |
| Danish | FLEURS 3,790, OpenSubtitles 200,000 | 0.00% | 1.86 | 0.44 | 3.83 | 0.46 | 0.0% |
| Bulgarian | FLEURS 4,026, OpenSubtitles 200,000 | 0.00% | 2.60 | 0.56 | 5.01 | 0.55 | 0.0% |
| Russian | FLEURS 3,693, OpenSubtitles 200,000 | 0.00% | 2.23 | 0.44 | 3.83 | 0.44 | 0.0% |
| Czech | FLEURS 3,839, OpenSubtitles 200,000 | 0.00% | 2.62 | 0.55 | 4.79 | 0.52 | 0.0% |
| Greek | FLEURS 4,136, OpenSubtitles 200,000 | 0.00% | 4.89 | 0.97 | 11.27 | 1.07 | 63.8% |
| Ukrainian | FLEURS 3,885, OpenSubtitles 200,000 | 0.00% | 2.85 | 0.57 | 5.55 | 0.57 | 0.0% |
| Spanish | FLEURS 4,112, OpenSubtitles 200,000 | 0.00% | 1.78 | 0.38 | 3.48 | 0.45 | 0.0% |
| Dutch | FLEURS 3,453, OpenSubtitles 200,000 | 0.00% | 1.76 | 0.40 | 4.32 | 0.53 | 0.0% |
| Slovak | FLEURS 3,101, OpenSubtitles 200,000 | 0.00% | 2.52 | 0.53 | 4.73 | 0.51 | 0.0% |
| Croatian | FLEURS 4,752, OpenSubtitles 200,000 | 0.00% | 2.31 | 0.48 | 4.08 | 0.46 | 0.1% |
| Polish | FLEURS 3,937, OpenSubtitles 200,000 | 0.00% | 2.28 | 0.43 | 3.98 | 0.45 | 0.0% |
| Lithuanian | FLEURS 4,339, OpenSubtitles 200,000 | 0.00% | 2.86 | 0.51 | 4.53 | 0.48 | 0.0% |
| Norwegian Bokmal | FLEURS 3,687, OpenSubtitles 200,000 | 0.00% | 1.84 | 0.43 | 3.35 | 0.40 | 0.0% |
| Norwegian Nynorsk | Wikipedia 30,000 (not spoken) | 0.02% | 2.36 | 0.42 | - | - | - |
| Persian | FLEURS 4,341, OpenSubtitles 200,000 | 0.00% | 2.56 | 0.66 | 4.60 | 0.49 | 0.2% |
| Slovenian | FLEURS 3,695, OpenSubtitles 200,000 | 0.00% | 2.32 | 0.49 | 4.47 | 0.50 | 0.1% |
| Gujarati | FLEURS 4,577 | 0.00% | 5.06 | 1.00 | 9.91 | 0.95 | 35.7% |
| Latvian | FLEURS 3,317, OpenSubtitles 200,000 | 0.00% | 2.94 | 0.56 | 5.03 | 0.52 | 0.0% |
| Italian | FLEURS 4,286, OpenSubtitles 200,000 | 0.00% | 1.86 | 0.39 | 3.52 | 0.44 | 0.0% |
| Occitan | FLEURS 4,804 | 0.00% | 2.04 | 0.39 | 3.13 | 0.37 | 2.4% |
| Nepali | FLEURS 4,363 | 0.00% | 5.58 | 0.96 | 8.34 | 0.79 | 10.4% |
| Marathi | FLEURS 4,727 | 0.00% | 5.84 | 0.97 | 8.08 | 0.76 | 9.2% |
| Belarusian | FLEURS 3,808 | 0.00% | 3.69 | 0.61 | 5.24 | 0.53 | 0.1% |
| Serbian | FLEURS 3,934, OpenSubtitles 200,000 | 0.00% | 2.33 | 0.51 | 4.59 | 0.49 | 0.0% |
| Luxembourgish | FLEURS 3,844 | 0.00% | 2.46 | 0.42 | 4.40 | 0.50 | 0.0% |
| Venetian | Wikipedia 30,000 (not spoken) | 0.01% | 2.01 | 0.44 | - | - | - |
| Assamese | FLEURS 4,214 | 4.56% | 5.69 | 1.02 | 7.97 | 0.75 | 13.7% |
| Welsh | FLEURS 4,895 | 0.00% | 2.37 | 0.49 | 4.12 | 0.47 | 12.1% |
| Silesian | Wikipedia 30,000 (not spoken) | 0.01% | 2.96 | 0.50 | - | - | - |
| Asturian | FLEURS 3,855 | 0.00% | 1.94 | 0.37 | 3.94 | 0.48 | 0.0% |
| Chhattisgarhi | none | - | - | - | - | - | - |
| Awadhi | none | - | - | - | - | - | - |
| Maithili | Wikipedia 30,000 (not spoken) | 0.00% | 4.87 | 0.95 | - | - | - |
| Bhojpuri | Wikipedia 30,000 (not spoken) | 0.13% | 3.91 | 0.92 | - | - | - |
| Sindhi | FLEURS 4,849 | 0.00% | 2.75 | 0.73 | 5.32 | 0.58 | 0.5% |
| Irish | FLEURS 4,056 | 0.00% | 2.40 | 0.49 | 3.90 | 0.44 | 0.0% |
| Faroese | Wikipedia 30,000 (not spoken) | 0.00% | 2.70 | 0.50 | - | - | - |
| Hindi | FLEURS 2,777, OpenSubtitles 143,563 | 0.28% | 3.63 | 0.97 | 8.55 | 0.85 | 21.4% |
| Punjabi | FLEURS 2,748 | 3.17% | 4.25 | 1.02 | 8.57 | 0.85 | 26.0% |
| Bengali | FLEURS 4,328, OpenSubtitles 200,000 | 5.11% | 4.75 | 1.02 | 8.24 | 0.77 | 8.2% |
| Oriya | FLEURS 2,356 | 0.70% | 5.89 | 1.00 | 9.67 | 0.91 | 27.9% |
| Tajik | FLEURS 3,138 | 0.00% | 3.39 | 0.60 | 5.23 | 0.54 | 0.6% |
| Eastern Yiddish | Wikipedia 30,000 (not spoken) | 0.09% | 2.91 | 0.61 | - | - | - |
| Lombard | Wikipedia 30,000 (not spoken) | 0.01% | 2.09 | 0.47 | - | - | - |
| Ligurian | Wikipedia 30,000 (not spoken) | 0.11% | 2.35 | 0.48 | - | - | - |
| Sicilian | Wikipedia 30,000 (not spoken) | 0.00% | 2.38 | 0.45 | - | - | - |
| Friulian | Wikipedia 24,143 (not spoken) | 0.01% | 2.00 | 0.43 | - | - | - |
| Sardinian | Wikipedia 30,000 (not spoken) | 0.01% | 2.18 | 0.43 | - | - | - |
| Galician | FLEURS 3,497, OpenSubtitles 200,000 | 0.00% | 1.88 | 0.40 | 3.89 | 0.48 | 0.0% |
| Catalan | FLEURS 3,644, OpenSubtitles 200,000 | 0.00% | 1.96 | 0.43 | 3.78 | 0.46 | 0.0% |
| Icelandic | FLEURS 1,008, OpenSubtitles 200,000 | 0.00% | 2.56 | 0.56 | 5.00 | 0.55 | 0.3% |
| Tosk Albanian | OpenSubtitles 200,000 | 0.00% | 2.33 | 0.53 | - | - | - |
| Limburgish | Wikipedia 30,000 (not spoken) | 0.01% | 2.35 | 0.44 | - | - | - |
| Dari | none | - | - | - | - | - | - |
| Afrikaans | FLEURS 1,494, OpenSubtitles 61,311 | 0.00% | 1.81 | 0.43 | 4.35 | 0.52 | 13.1% |
| Macedonian | FLEURS 3,725, OpenSubtitles 200,000 | 0.00% | 2.63 | 0.56 | 5.13 | 0.56 | 0.0% |
| Sinhala | OpenSubtitles 200,000 | 0.00% | 4.76 | 0.98 | - | - | - |
| Urdu | FLEURS 2,675, OpenSubtitles 46,523 | 0.08% | 2.94 | 0.83 | 6.88 | 0.74 | 5.4% |
| Magahi | none | - | - | - | - | - | - |
| Bosnian | FLEURS 4,416, OpenSubtitles 200,000 | 0.00% | 2.27 | 0.49 | 4.09 | 0.46 | 0.0% |
| Armenian | FLEURS 4,380, OpenSubtitles 4,092 | 0.00% | 6.27 | 1.00 | 10.30 | 0.95 | 41.4% |
| Simplified Chinese | FLEURS 4,600, OpenSubtitles 200,000 | 0.01% | 5.18 | 0.70 | 2.44 | 0.21 | 0.0% |
| Traditional Chinese | OpenSubtitles 200,000 | 0.54% | 5.52 | 0.80 | - | - | - |
| Cantonese | FLEURS 3,120 | 0.03% | 16.77 | 0.78 | 2.27 | 0.19 | 0.0% |
| Burmese | FLEURS 4,322 | 0.00% | 14.44 | 0.99 | 9.93 | 0.85 | 24.2% |
| Standard Arabic | FLEURS 2,827, OpenSubtitles 200,000 | 0.00% | 2.11 | 0.49 | 3.64 | 0.43 | 0.1% |
| Najdi Arabic | none | - | - | - | - | - | - |
| Levantine Arabic | none | - | - | - | - | - | - |
| Egyptian Arabic | Wikipedia 30,000 (not spoken) | 0.07% | 2.20 | 0.49 | - | - | - |
| Moroccan Arabic | Wikipedia 30,000 (not spoken) | 0.04% | 2.52 | 0.55 | - | - | - |
| Mesopotamian Arabic | none | - | - | - | - | - | - |
| Taizzi-Adeni Arabic | none | - | - | - | - | - | - |
| Tunisian Arabic | none | - | - | - | - | - | - |
| Hebrew | FLEURS 4,362, OpenSubtitles 200,000 | 0.02% | 2.05 | 0.48 | 3.64 | 0.42 | 0.3% |
| Maltese | FLEURS 4,225 | 0.00% | 3.09 | 0.47 | 4.48 | 0.47 | 0.0% |
| Indonesian | FLEURS 3,616, OpenSubtitles 200,000 | 0.00% | 2.14 | 0.39 | 3.19 | 0.38 | 0.0% |
| Malay | FLEURS 3,740, OpenSubtitles 200,000 | 0.00% | 2.12 | 0.39 | 3.37 | 0.39 | 0.0% |
| Tagalog | FLEURS 3,266, OpenSubtitles 19,314 | 0.00% | 2.12 | 0.42 | 3.39 | 0.40 | 0.0% |
| Cebuano | FLEURS 4,027 | 0.00% | 1.95 | 0.38 | 3.55 | 0.43 | 0.1% |
| Javanese | FLEURS 4,074 | 0.00% | 2.37 | 0.40 | 3.39 | 0.39 | 0.0% |
| Sundanese | Wikipedia 30,000 (not spoken) | 0.05% | 2.21 | 0.37 | - | - | - |
| Minangkabau | Wikipedia 30,000 (not spoken) | 0.06% | 2.31 | 0.39 | - | - | - |
| Balinese | Wikipedia 30,000 (not spoken) | 1.11% | 2.49 | 0.42 | - | - | - |
| Banjar | Wikipedia 30,000 (not spoken) | 0.02% | 2.41 | 0.38 | - | - | - |
| Pangasinan | Wikipedia 13,534 (not spoken) | 0.01% | 1.82 | 0.36 | - | - | - |
| Iloko | Wikipedia 30,000 (not spoken) | 0.12% | 2.12 | 0.41 | - | - | - |
| Waray | Wikipedia 30,000 (not spoken) | 0.00% | 2.00 | 0.39 | - | - | - |
| Tamil | FLEURS 3,335, OpenSubtitles 40,165 | 0.00% | 6.95 | 0.99 | 10.01 | 0.90 | 36.2% |
| Telugu | FLEURS 3,085, OpenSubtitles 30,416 | 0.00% | 6.02 | 0.98 | 8.87 | 0.81 | 15.8% |
| Kannada | FLEURS 3,489 | 0.00% | 7.59 | 1.00 | 9.13 | 0.83 | 8.9% |
| Malayalam | FLEURS 4,419, OpenSubtitles 200,000 | 0.00% | 7.82 | 0.99 | 10.16 | 0.90 | 36.6% |
| Turkish | FLEURS 3,607, OpenSubtitles 200,000 | 0.00% | 2.44 | 0.41 | 3.34 | 0.38 | 0.0% |
| North Azerbaijani | FLEURS 3,988 | 0.00% | 3.48 | 0.53 | 5.10 | 0.53 | 0.0% |
| Northern Uzbek | FLEURS 4,168 | 0.00% | 3.14 | 0.43 | 4.47 | 0.47 | 0.0% |
| Kazakh | FLEURS 4,425, OpenSubtitles 4,100 | 0.00% | 4.43 | 0.67 | 5.39 | 0.53 | 0.0% |
| Bashkir | Wikipedia 30,000 (not spoken) | 0.01% | 4.38 | 0.68 | - | - | - |
| Tatar | Wikipedia 30,000 (not spoken) | 0.01% | 3.88 | 0.61 | - | - | - |
| Thai | FLEURS 4,062, OpenSubtitles 200,000 | 0.00% | 5.51 | 0.37 | 3.14 | 0.28 | 0.0% |
| Lao | FLEURS 2,405 | 0.00% | 19.80 | 0.96 | 7.93 | 0.67 | 8.1% |
| Finnish | FLEURS 4,037, OpenSubtitles 200,000 | 0.00% | 2.82 | 0.46 | 4.14 | 0.43 | 0.0% |
| Estonian | FLEURS 3,781, OpenSubtitles 200,000 | 0.00% | 2.44 | 0.47 | 4.25 | 0.46 | 0.0% |
| Hungarian | FLEURS 4,407, OpenSubtitles 200,000 | 0.00% | 2.69 | 0.49 | 4.89 | 0.52 | 0.0% |
| Vietnamese | FLEURS 4,212, OpenSubtitles 200,000 | 0.47% | 1.44 | 0.42 | 3.26 | 0.47 | 0.0% |
| Khmer | FLEURS 2,772 | 0.00% | 24.91 | 0.97 | 9.22 | 0.77 | 8.3% |
| Japanese | FLEURS 3,208, OpenSubtitles 200,000 | 0.01% | 5.25 | 0.70 | 2.87 | 0.24 | 0.0% |
| Korean | FLEURS 2,915, OpenSubtitles 200,000 | 0.01% | 2.38 | 0.81 | 3.06 | 0.34 | 0.0% |
| Georgian | FLEURS 2,879, OpenSubtitles 200,000 | 0.00% | 4.71 | 0.99 | 10.26 | 0.93 | 36.7% |
| Basque | OpenSubtitles 200,000 | 0.00% | 2.60 | 0.45 | - | - | - |
| Haitian | Wikipedia 30,000 (not spoken) | 0.00% | 2.18 | 0.48 | - | - | - |
| Papiamento | Wikipedia 20,427 (not spoken) | 0.00% | 1.92 | 0.42 | - | - | - |
| Kabuverdianu | FLEURS 3,945 | 0.00% | 2.01 | 0.42 | 3.15 | 0.38 | 0.0% |
| Tok Pisin | Wikipedia 3,215 (not spoken) | 0.18% | 2.03 | 0.43 | - | - | - |
| Swahili | FLEURS 3,768 | 0.00% | 2.44 | 0.44 | 3.20 | 0.36 | 0.0% |

[spoken measurement] 644 s since the start


### Word timings with Whisper (KsponSpeech and FLEURS)

Real word onsets are measured with Whisper large-v3 (S2.3): the reference transcript is teacher forced through the decoder and
dynamic time warping over the cross attention of its alignment heads gives each word its onset (`whisper_alignment`). The data
is KsponSpeech's evaluation split (6,000 utterances, spoken form of the transcript) and, for every language with FLEURS, the
first 200 utterances of its `dev` split. Languages Whisper lacks are prompted with a close one. The job runs on a Kaggle GPU
through letify: KsponSpeech travels through the Drive store, FLEURS is read from the hub on the remote machine, and the result
is saved as `word_timings.json` in the output folder, which later runs reuse. Delete that file to measure again.

In [10]:
import importlib.metadata
import zipfile

WORD_TIMINGS = OUT / "word_timings.json"
KSPONSPEECH = Path(".cache/datasets/ksponspeech")
WORD_TIMING_DEVICE = ("kaggle_60", "T4")
WORD_TIMING_UTTERANCES = 200
# Whisper's token where it differs from the FLEURS prefix, and a close language where Whisper has none.
WHISPER_LANGUAGE = {"cmn": "zh", "fil": "tl", "nb": "no", "jv": "jw", "ast": "es", "ga": "en", "or": "bn", "ceb": "tl", "kea": "pt"}
# Packages the remote machine installs beyond the lock, pinned to the versions here.
REMOTE_PACKAGES = ("sentencepiece", "protobuf", "pyloudnorm", "pyarrow", "librosa", "soundfile")


def whisper_language(fleurs_code):
    prefix = fleurs_code.split("_")[0]
    return WHISPER_LANGUAGE.get(prefix, prefix)


def package_zip():
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path("polyplex_research").rglob("*.py")):
            archive.write(path, str(path))
    return buffer.getvalue()


def whisper_aligner(model, processor, whisper_alignment):
    # Returns align(clips, language) for clips of (id, transcript, 16 kHz waveform), one row per clip kept.
    import torch

    def align_chunk(chunk, language):
        # The cross attention of every layer is kept, so a batch of long transcripts can exceed the card; it is
        # then split in half until it fits. The split runs after the except block, once the failed batch's
        # tensors are released.
        try:
            return whisper_alignment(model, processor, [c[2] for c in chunk], [c[1] for c in chunk], language=language)
        except torch.OutOfMemoryError:
            if len(chunk) == 1:
                raise
        torch.cuda.empty_cache()
        half = len(chunk) // 2
        return align_chunk(chunk[:half], language) + align_chunk(chunk[half:], language)

    def decoder_length(text):
        # Prompt (start, language, task, no timestamps), the word tokens and the end token, as whisper_alignment builds it.
        return 4 + sum(len(processor.tokenizer.encode(" " + word, add_special_tokens=False)) for word in text.split()) + 1

    def align(clips, language):
        # Whisper reads at most 30 seconds and its decoder at most max_target_positions tokens; a longer transcript
        # indexes past the position table and breaks the CUDA context. A clip without words has nothing to place.
        limit = model.config.max_target_positions
        kept = [clip for clip in clips
                if clip[1].split() and len(clip[2]) <= 30 * 16000 and decoder_length(clip[1]) <= limit]
        if len(kept) < len(clips):
            print(f"{language}: {len(clips) - len(kept)} of {len(clips)} clips left out (empty, over 30 s or over {limit} tokens)", flush=True)
        clips = kept
        rows = []
        for start in range(0, len(clips), 8):
            chunk = clips[start:start + 8]
            for (clip_id, text, waveform), (starts, ends, loss) in zip(chunk, align_chunk(chunk, language)):
                rows.append({"id": clip_id, "text": text, "seconds": len(waveform) / 16000,
                             "starts": [float(s) for s in starts], "ends": [float(e) for e in ends], "loss": float(loss)})
        return rows

    return align


def remote_word_timings(package, fleurs_codes, languages, per_language, ksponspeech):
    # Runs on the remote GPU, where the package arrives as a zip.
    import io
    import json
    import os
    import sys
    import tarfile
    import tempfile
    import zipfile

    import letify
    import numpy as np
    import soundfile
    import torch
    from huggingface_hub import hf_hub_download
    from transformers import WhisperForConditionalGeneration, WhisperProcessor

    root = tempfile.mkdtemp(prefix="package-")
    zipfile.ZipFile(io.BytesIO(package)).extractall(root)
    sys.path.insert(0, root)
    from polyplex_research.datasets.duplex_data import whisper_alignment
    from polyplex_research.datasets.ksponspeech import normalize_transcription

    processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
    model = WhisperForConditionalGeneration.from_pretrained(
        "openai/whisper-large-v3", torch_dtype=torch.float16, attn_implementation="eager").to("cuda").eval()

    align = whisper_aligner(model, processor, whisper_alignment)

    def save(result):
        # On the Drive store after every source, for a connection lost on the way back. The call's own return
        # value carries the result, so a Drive that cannot be reached only costs this copy.
        path = os.path.join(tempfile.mkdtemp(prefix="word-timings-"), "word_timings.json")
        with open(path, "w", encoding="utf-8") as sink:
            json.dump(result, sink, ensure_ascii=False)
        try:
            letify.stash("word-timings-result", path)
        except Exception as error:
            print("Drive copy failed:", type(error).__name__, error, flush=True)

    def ksponspeech_rows():
        data = letify.unstash(ksponspeech, tempfile.mkdtemp(prefix="ksponspeech-"))
        clips = []
        for split in ("eval_clean", "eval_other"):
            for line in open(os.path.join(data, f"{split}.trn"), encoding="utf-8"):
                path, text = line.rstrip("\n").split(" :: ", 1)
                name = os.path.basename(path)
                samples = np.fromfile(os.path.join(data, split, name), dtype="<i2").astype(np.float32) / 32768
                clips.append((name.rsplit(".", 1)[0], normalize_transcription(text, "phonetic"), samples))
        return align(clips, "ko")

    def fleurs_rows(code):
        transcripts = hf_hub_download("google/fleurs", f"data/{code}/dev.tsv", repo_type="dataset")
        audio = hf_hub_download("google/fleurs", f"data/{code}/audio/dev.tar.gz", repo_type="dataset")
        wanted = {}
        for line in open(transcripts, encoding="utf-8"):
            fields = line.rstrip("\n").split("\t")
            if len(fields) >= 3 and fields[1] not in wanted and len(wanted) < per_language:
                wanted[fields[1]] = fields[2]
        waveforms = {}
        with tarfile.open(audio, "r:gz") as archive:
            for member in archive:
                name = os.path.basename(member.name)
                if member.isfile() and name in wanted:
                    samples, _ = soundfile.read(io.BytesIO(archive.extractfile(member).read()), dtype="float32")
                    waveforms[name] = np.asarray(samples, dtype=np.float32).reshape(-1)
        # One archive on disk at a time.
        os.remove(os.path.realpath(audio))
        clips = [(name.rsplit(".", 1)[0], text, waveforms[name]) for name, text in wanted.items() if name in waveforms]
        return align(clips, languages[code])

    # A source that fails is recorded and skipped, so the call returns everything that did finish.
    result, failed = {}, {}
    jobs = ([("ksponspeech", ksponspeech_rows)] if ksponspeech else []) + [(code, lambda code=code: fleurs_rows(code)) for code in fleurs_codes]
    for name, job in jobs:
        try:
            result[name] = job()
        except Exception as error:
            failed[name] = f"{type(error).__name__}: {error}"
            torch.cuda.empty_cache()
            print(name, "failed:", failed[name], flush=True)
            continue
        save(result)
        print(name, len(result[name]), flush=True)
    return {"timings": result, "failed": failed}


WORD_TIMINGS_PARTIAL = OUT / "word_timings_partial.json"

if not WORD_TIMINGS.exists():
    import letify

    # What earlier runs finished is kept in the partial file and not measured again.
    done = json.load(open(WORD_TIMINGS_PARTIAL)) if WORD_TIMINGS_PARTIAL.exists() else {}
    launcher = letify.Launcher()
    alias, accelerator = WORD_TIMING_DEVICE
    device = getattr(getattr(launcher.providers, alias), accelerator)
    env = letify.Env.from_lock().pip_install(*(f"{name}=={importlib.metadata.version(name)}" for name in REMOTE_PACKAGES))
    # One attempt per call: a retry would start every source again, and this cell already skips finished ones.
    run = launcher.function(device=device, host=letify.remote, env=env, retries=1)(remote_word_timings)
    if "ksponspeech" not in done:
        letify.stash("ksponspeech-eval", KSPONSPEECH)
    codes = [language.fleurs for language in QWEN3_LANGUAGES if language.fleurs and language.fleurs not in done]
    try:
        returned = run(package_zip(), codes, {code: whisper_language(code) for code in codes}, WORD_TIMING_UTTERANCES,
                       None if "ksponspeech" in done else "ksponspeech-eval")
    except Exception:
        # The call itself was lost; the Drive store holds every source it finished.
        partial = letify.unstash("word-timings-result", OUT / "word_timings_remote.json")
        if partial is not None:
            done.update(json.load(open(partial)))
            json.dump(done, open(WORD_TIMINGS_PARTIAL, "w"), ensure_ascii=False)
        raise
    done.update(returned["timings"])
    json.dump(done, open(WORD_TIMINGS_PARTIAL, "w"), ensure_ascii=False)
    if returned["failed"]:
        raise RuntimeError(f"Word timings failed for {returned['failed']}; run this cell again to measure only those.")
    json.dump(done, open(WORD_TIMINGS, "w"), ensure_ascii=False)
word_timings = json.load(open(WORD_TIMINGS))
print(f"Word timings: {len(word_timings.get('ksponspeech', [])):,} KsponSpeech utterances, "
      f"{sum(len(rows) for name, rows in word_timings.items() if name != 'ksponspeech'):,} FLEURS utterances in "
      f"{len(word_timings) - ('ksponspeech' in word_timings)} languages")

Word timings: 6,000 KsponSpeech utterances, 15,257 FLEURS utterances in 79 languages


### Word timings with Whisper on real speech (YODAS2)

FLEURS is read speech, and read speech leaves longer gaps between words than conversation, so it understates the cut rate
(S2.3). Real speech for every language comes from YODAS2 (`espnet/yodas2`, YouTube videos under CC BY, about 85 of the
Qwen3 languages), the user-uploaded subtitle subsets (`000`). For each language the subtitle JSON of the first shard is read,
and the audio shard is streamed from its start until enough videos are read, so the 1.5 GB shard is never downloaded whole.
Subtitle boundaries can be off, so the neighbouring subtitle segments of a video are joined into pieces of at most 25 seconds,
split at pauses over one second, with 0.3 seconds of margin on both sides; a boundary error then only touches the two ends of a
piece. Each piece is aligned with Whisper against its subtitle text, and the alignment loss is kept so pieces whose subtitles
do not match the speech can be left out. Korean was measured first and compared with KsponSpeech. Every language with a user-subtitle subset gets 1,000 pieces, 250
from each of four shards spread over the subset so one channel does not fill the sample. The languages are split into one group per
device and each group runs in its own process, so the devices work at the same time: the department server's card 0 (a Tesla
P100, which the CUDA 12.8 build of PyTorch does not support, so that environment is synced again with the `cu126` group) and
the T4s of two Kaggle accounts. Each device group runs in its own Python process that reads this notebook's cells, because the
department server's call did not start when one process drove several devices, from threads or from forked children, while
the same call alone went through.

In [11]:
YODAS_TIMINGS = OUT / "yodas_timings.json"
YODAS_TIMINGS_PARTIAL = OUT / "yodas_timings_partial.json"
# The CUDA 12.8 build of PyTorch has no kernels for the P100 (sm_60), so its environment is synced again with the cu126
# group after letify's default sync. The sync is exact so the cu128-only packages go, and the packages letify installed on
# top of the lock are installed again after it.
UV_ON_REMOTE = ("UV_PYTHON_INSTALL_DIR=/workspace/uv-python UV_CACHE_DIR=/workspace/uv-cache "
                "\"$(command -v uv || echo /workspace/uv-bin/uv)\"")
CU126_SYNC = (f"{UV_ON_REMOTE} sync --frozen --no-install-project --no-group cu128 --group cu126 --python 3.13",
              f"{UV_ON_REMOTE} pip install --python .venv/bin/python "
              + " ".join(f"{name}=={importlib.metadata.version(name)}" for name in REMOTE_PACKAGES))
# One group of languages per device: (provider, accelerator, extra environment commands).
YODAS_DEVICES = [("dept_gpu", "Tesla_P100", CU126_SYNC), ("kaggle_60", "T4", ()), ("kaggle_30", "T4", ())]
YODAS_PIECES = 1000
YODAS_SHARDS = 4
# YODAS2 code where it is not the Wikipedia code of the table, and the Whisper token where Whisper spells the code
# differently or lacks the language (a close language prompts the forced alignment).
YODAS_CODE = {"Hebrew": "iw", "Bhojpuri": "bh", "Simplified Chinese": "zh", "Norwegian Bokmal": "no"}
YODAS_WHISPER = {"iw": "he", "jv": "jw", "bh": "hi", "ga": "en", "or": "bn", "sc": "it"}


def yodas_subsets():
    from huggingface_hub import HfApi

    folders = {f.path.split("/")[1] for f in HfApi().list_repo_tree("espnet/yodas2", path_in_repo="data", repo_type="dataset")}
    subsets = {}
    for language in QWEN3_LANGUAGES:
        code = YODAS_CODE.get(language.name, language.wikipedia)
        # The 000 subsets carry the subtitles users uploaded.
        if code and f"{code}000" in folders and f"{code}000" not in subsets:
            subsets[f"{code}000"] = YODAS_WHISPER.get(code, code)
    return subsets


YODAS_SUBSETS = yodas_subsets()


def remote_yodas_timings(package, subsets, pieces_per_language, shards, stash_name):
    # Runs on the remote GPU, where the package arrives as a zip.
    import io
    import json
    import os
    import sys
    import tarfile
    import tempfile
    import zipfile

    import letify
    import librosa
    import numpy as np
    import soundfile
    import torch
    from huggingface_hub import HfFileSystem, hf_hub_download
    from transformers import WhisperForConditionalGeneration, WhisperProcessor

    root = tempfile.mkdtemp(prefix="package-")
    zipfile.ZipFile(io.BytesIO(package)).extractall(root)
    sys.path.insert(0, root)
    from polyplex_research.datasets.duplex_data import whisper_alignment

    processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
    model = WhisperForConditionalGeneration.from_pretrained(
        "openai/whisper-large-v3", torch_dtype=torch.float16, attn_implementation="eager").to("cuda").eval()
    align = whisper_aligner(model, processor, whisper_alignment)
    fs = HfFileSystem()

    def pieces_of(audio_id, segments, waveform):
        # Subtitle ids end in -<index>-<start>-<end>, the times in hundredths of a second.
        timed = []
        for utt_id, text in segments.items():
            _, _, start, end = utt_id.rsplit("-", 3)
            if text.split():
                timed.append((int(start) / 100, int(end) / 100, " ".join(text.split())))
        timed.sort()
        pieces, current = [], []

        def flush():
            if current:
                begin = max(0.0, current[0][0] - 0.3)
                finish = min(len(waveform) / 16000, current[-1][1] + 0.3)
                clip = waveform[int(begin * 16000):int(finish * 16000)]
                pieces.append((f"{audio_id}-{begin:.2f}", " ".join(t for _, _, t in current), clip))

        for segment in timed:
            if current and (segment[1] - current[0][0] > 25 or segment[0] - current[-1][1] > 1.0):
                flush()
                current = []
            current.append(segment)
        flush()
        return pieces

    def shard_rows(subset, shard, language, wanted):
        videos = json.load(open(hf_hub_download("espnet/yodas2", f"data/{subset}/text/{shard}.json", repo_type="dataset")))
        segments = {video["audio_id"]: video["text"] for video in videos}
        rows = []
        # Streamed from the start and closed once enough pieces are aligned, so only a prefix of the shard is read.
        with fs.open(f"datasets/espnet/yodas2/data/{subset}/audio/{shard}.tar.gz", "rb", block_size=4 * 2**20) as raw:
            with tarfile.open(fileobj=raw, mode="r|gz") as archive:
                for member in archive:
                    audio_id = os.path.basename(member.name).rsplit(".", 1)[0]
                    if not member.isfile() or audio_id not in segments:
                        continue
                    samples, rate = soundfile.read(io.BytesIO(archive.extractfile(member).read()), dtype="float32")
                    if samples.ndim > 1:
                        samples = samples.mean(axis=1)
                    waveform = librosa.resample(samples, orig_sr=rate, target_sr=16000) if rate != 16000 else samples
                    rows += align(pieces_of(audio_id, segments[audio_id], waveform), language)
                    if len(rows) >= wanted:
                        break
        return rows[:wanted]

    def subset_rows(subset, language):
        names = sorted(os.path.basename(f["name"]).rsplit(".", 1)[0]
                       for f in fs.ls(f"datasets/espnet/yodas2/data/{subset}/text", detail=True))
        # Shards spread evenly over the subset, each giving an equal share of the pieces.
        picked = [names[round(k * (len(names) - 1) / max(shards - 1, 1))] for k in range(min(shards, len(names)))]
        picked = list(dict.fromkeys(picked))
        rows = []
        for k, shard in enumerate(picked):
            wanted = (pieces_per_language - len(rows)) // (len(picked) - k)
            rows += shard_rows(subset, shard, language, wanted)
        return rows

    def save(result):
        path = os.path.join(tempfile.mkdtemp(prefix="yodas-timings-"), "yodas_timings.json")
        with open(path, "w", encoding="utf-8") as sink:
            json.dump(result, sink, ensure_ascii=False)
        try:
            letify.stash(stash_name, path)
        except Exception as error:
            print("Drive copy failed:", type(error).__name__, error, flush=True)

    # A subset that fails is recorded and skipped, so the call returns everything that did finish.
    result, failed = {}, {}
    for subset, language in subsets.items():
        try:
            result[subset] = subset_rows(subset, language)
        except Exception as error:
            failed[subset] = f"{type(error).__name__}: {error}"
            torch.cuda.empty_cache()
            print(subset, "failed:", failed[subset], flush=True)
            continue
        save(result)
        print(subset, len(result[subset]), flush=True)
    return {"timings": result, "failed": failed}


def complete(timings):
    # A subset with a result is done; one with fewer than YODAS_PIECES pieces ran out of videos in its shards, which a
    # second run would not change.
    return {subset: rows for subset, rows in timings.items() if rows}


def run_group(k, group):
    # Runs in its own forked process with its own letify launcher; returns what the group finished and what failed.
    import letify

    alias, accelerator, commands = YODAS_DEVICES[k]
    env = letify.Env.from_lock().pip_install(*(f"{name}=={importlib.metadata.version(name)}" for name in REMOTE_PACKAGES))
    if commands:
        env = env.run(*commands)
    launcher = letify.Launcher()
    device = getattr(getattr(launcher.providers, alias), accelerator)
    run = launcher.function(device=device, host=letify.remote, env=env, retries=1)(remote_yodas_timings)
    stash_name = f"yodas-timings-{alias}"
    try:
        returned = run(package_zip(), group, YODAS_PIECES, YODAS_SHARDS, stash_name)
        return returned["timings"], returned["failed"]
    except Exception as error:
        partial = letify.unstash(stash_name, OUT / f"yodas_timings_remote_{alias}.json")
        timings = json.load(open(partial)) if partial is not None else {}
        return timings, {subset: f"{type(error).__name__}: {error}" for subset in group if subset not in timings}


# A child interpreter runs one device group: it executes the notebook cells that define the language table, the Whisper
# aligner and this cell's functions, then writes the group's result to a file.
GROUP_RUNNER = """
import io, json, sys, nbformat, __main__
cells = nbformat.read("build_vocab.ipynb", 4).cells
names = __main__.__dict__
names["io"] = io
for marker in ("QWEN3_LANGUAGES = (", "def whisper_aligner(", "def run_group("):
    source = next(cell.source for cell in cells if cell.cell_type == "code" and marker in cell.source)
    exec(source[:source.index("\\nif not YODAS_TIMINGS.exists():")] if marker == "def run_group(" else source, names)
timings, failed = names["run_group"](int(sys.argv[1]), json.loads(sys.argv[2]))
json.dump({"timings": timings, "failed": failed}, open(sys.argv[3], "w"), ensure_ascii=False)
"""


if not YODAS_TIMINGS.exists():
    import subprocess
    import sys
    import time

    done = complete(json.load(open(YODAS_TIMINGS_PARTIAL))) if YODAS_TIMINGS_PARTIAL.exists() else {}
    todo = [subset for subset in YODAS_SUBSETS if subset not in done]
    groups = [{subset: YODAS_SUBSETS[subset] for subset in todo[k::len(YODAS_DEVICES)]} for k in range(len(YODAS_DEVICES))]
    children = {}
    for k, group in enumerate(groups):
        if group:
            alias = YODAS_DEVICES[k][0]
            result = OUT / f"yodas_timings_{alias}.json"
            result.unlink(missing_ok=True)
            log = open(OUT / f"yodas_timings_{alias}.log", "w")
            children[k] = (subprocess.Popen([sys.executable, "-c", GROUP_RUNNER, str(k), json.dumps(group), str(result)],
                                            stdout=log, stderr=subprocess.STDOUT), result)
    failed = {}
    # Each group's result is merged and saved as soon as its process ends.
    while children:
        for k, (child, result) in list(children.items()):
            if child.poll() is None:
                continue
            returned = json.load(open(result)) if result.exists() else {"timings": {}, "failed": {s: f"process exited {child.returncode}" for s in groups[k]}}
            done.update(returned["timings"])
            failed.update(returned["failed"])
            json.dump(done, open(YODAS_TIMINGS_PARTIAL, "w"), ensure_ascii=False)
            del children[k]
        time.sleep(30)
    if failed:
        raise RuntimeError(f"YODAS2 timings failed for {failed}; run this cell again to measure only those.")
    json.dump(done, open(YODAS_TIMINGS, "w"), ensure_ascii=False)
yodas_timings = json.load(open(YODAS_TIMINGS))
print({subset: len(rows) for subset, rows in yodas_timings.items()})

{'pt000': 972, 'sv000': 1000, 'ru000': 1000, 'uk000': 913, 'sk000': 1000, 'lt000': 1000, 'sl000': 1000, 'it000': 1000, 'mr000': 575, 'lb000': 9, 'bh000': 1, 'fo000': 10, 'bn000': 472, 'yi000': 8, 'ca000': 1000, 'af000': 72, 'ur000': 638, 'zh000': 860, 'iw000': 1000, 'jv000': 202, 'te000': 698, 'tr000': 1000, 'kk000': 458, 'th000': 883, 'et000': 1000, 'km000': 444, 'ka000': 1000, 'sw000': 633, 'en000': 1000, 'de000': 785, 'da000': 1000, 'cs000': 1000, 'es000': 766, 'hr000': 1000, 'no000': 838, 'gu000': 465, 'oc000': 28, 'be000': 1000, 'as000': 104, 'sd000': 13, 'hi000': 1000, 'or000': 297, 'sc000': 2, 'is000': 654, 'mk000': 787, 'bs000': 801, 'my000': 299, 'id000': 1000, 'su000': 13, 'kn000': 591, 'az000': 799, 'ba000': 14, 'lo000': 25, 'hu000': 1000, 'ja000': 770, 'eu000': 1000, 'fr000': 1000, 'ro000': 1000, 'bg000': 753, 'el000': 1000, 'nl000': 1000, 'pl000': 994, 'fa000': 797, 'lv000': 328, 'ne000': 556, 'sr000': 1000, 'cy000': 487, 'ga000': 182, 'pa000': 467, 'tg000': 47, 'gl000': 5